In [ ]:
# Outlier analysis for `ABVERKAUFTE_MENGE_KG`

Goal: identify and quantify outliers in the daily aggregated transactions before model training.

`ABVERKAUFTE_MENGE_KG` is the sold quantity per (`ARTIKEL_ID`, `MARKT_ID`, `DATE`). A single global distribution is hard to interpret because the column mixes physically different quantity meanings. For descriptive distribution plots, daily observations are therefore split by `GEWICHT_FLAG`:

- `GEWICHT_FLAG = 1`: articles whose `ARTIKEL_INHALT` contains `amm`, i.e. gram/kilogram package contents,
- `GEWICHT_FLAG = 0`: all other articles, if present in the input data.

Outliers are still detected **per (`ARTIKEL_ID`, `MARKT_ID`) series** with the Tukey upper-extreme rule:

- compute `Q1`, `Q3`, `IQR = Q3 - Q1` for each series,
- treat `ABVERKAUFTE_MENGE_KG > Q3 + 3 * IQR` as an extreme upper outlier and drop it.

The corresponding cleaning script is `src/data/cleaning/remove_outliers.py`.

**Why no lower bound.** We do not apply `Q1 - k * IQR`. Sold quantities are bounded below at 0, so the symmetric Tukey lower bound is rarely useful. Tiny positive residues from sale + return cancellation in float64 are already snapped to `0.0` in the daily aggregation step; anything else in the lower tail is legitimate low-demand signal and must be kept.

In [ ]:
from pathlib import Path
import os

os.environ.setdefault("MPLCONFIGDIR", "/tmp/matplotlib")
Path(os.environ["MPLCONFIGDIR"]).mkdir(parents=True, exist_ok=True)

import duckdb
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 80)
pd.set_option("display.float_format", lambda x: f"{x:,.6f}")
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (9, 5)

DEMAND_COL = "ABVERKAUFTE_MENGE_KG"
GROUP_COLS = ["ARTIKEL_ID", "MARKT_ID"]
IQR_K = 3.0

candidate_dirs = [
    Path("../../data/interim/transactions_daily_agg_fcm"),
    Path("../data/interim/transactions_daily_agg_fcm"),
    Path("data/interim/transactions_daily_agg_fcm"),
    Path("../../data/interim/transactions_daily_agg"),
    Path("../data/interim/transactions_daily_agg"),
    Path("data/interim/transactions_daily_agg"),
]
DATA_DIR = next(
    (p for p in candidate_dirs if sorted(p.glob("transactions_year_*.parquet"))),
    None,
)
if DATA_DIR is None:
    raise FileNotFoundError("Could not find daily aggregate parquet files.")
DATA_VARIANT = "FCM" if DATA_DIR.name.endswith("_fcm") else "base"
NO_OUTLIER_DIR_NAME = (
    "transactions_daily_agg_fcm_no_outliers"
    if DATA_VARIANT == "FCM"
    else "transactions_daily_agg_no_outliers"
)

PARQUET_GLOB = str(DATA_DIR / "*.parquet")
print(f"Reading daily aggregates from {DATA_DIR} ({DATA_VARIANT})")

con = duckdb.connect()
con.execute("PRAGMA threads=8")

required_daily_cols = {
    "ARTIKEL_ID",
    "MARKT_ID",
    "DATE",
    DEMAND_COL,
    "GEWICHT_FLAG",
}
daily_cols = {
    row[0]
    for row in con.execute(
        f"DESCRIBE SELECT * FROM read_parquet('{PARQUET_GLOB}')"
    ).fetchall()
}
missing_daily_cols = sorted(required_daily_cols - daily_cols)
if missing_daily_cols:
    raise ValueError(f"Daily aggregate files are missing columns: {missing_daily_cols}")

UNIT_GROUP_ORDER = ["gewicht_flag_1", "gewicht_flag_0"]
UNIT_GROUP_LABELS = {
    "gewicht_flag_1": "Gewichtsartikel (GEWICHT_FLAG = 1)",
    "gewicht_flag_0": "Nicht-Gewichtsartikel (GEWICHT_FLAG = 0)",
}

def fmt_int(v):
    return f"{int(v):,}"


def fmt_float(v, decimals=6):
    if v is None or (isinstance(v, float) and np.isnan(v)):
        return ""
    return f"{float(v):,.{decimals}f}"


## Distribution by `GEWICHT_FLAG`

A single distribution of `ABVERKAUFTE_MENGE_KG` mixes article groups whose quantities mean different things. For data understanding, the distribution is therefore shown separately by `GEWICHT_FLAG`.

The split is descriptive. The outlier filter below is still calculated per article-market series.

In [ ]:
raw = con.execute(f"""
SELECT
    COUNT(*)                                              AS n_rows,
    COUNT(DISTINCT (ARTIKEL_ID, MARKT_ID))                AS n_series,
    MIN({DEMAND_COL})                                     AS min,
    quantile_cont({DEMAND_COL}, 0.50)                     AS p50,
    quantile_cont({DEMAND_COL}, 0.90)                     AS p90,
    quantile_cont({DEMAND_COL}, 0.99)                     AS p99,
    quantile_cont({DEMAND_COL}, 0.999)                    AS p999,
    quantile_cont({DEMAND_COL}, 0.9999)                   AS p9999,
    MAX({DEMAND_COL})                                     AS max,
    AVG({DEMAND_COL})                                     AS mean,
    stddev_samp({DEMAND_COL})                             AS sd
FROM read_parquet('{PARQUET_GLOB}')
""").fetchone()

metric_meta = [
    ("n_rows",   "Total daily product-store rows",        "rows",     fmt_int),
    ("n_series", "Distinct (ARTIKEL_ID, MARKT_ID) series","series",   fmt_int),
    ("min",      "Smallest observed quantity",            "qty",      lambda v: fmt_float(v, 6)),
    ("p50",      "Median (50th percentile)",              "qty",      lambda v: fmt_float(v, 4)),
    ("p90",      "90th percentile",                       "qty",      lambda v: fmt_float(v, 4)),
    ("p99",      "99th percentile",                       "qty",      lambda v: fmt_float(v, 4)),
    ("p999",     "99.9th percentile",                     "qty",      lambda v: fmt_float(v, 4)),
    ("p9999",    "99.99th percentile",                    "qty",      lambda v: fmt_float(v, 4)),
    ("max",      "Largest single value (tail extreme)",   "qty",      lambda v: fmt_float(v, 4)),
    ("mean",     "Mean",                                  "qty",      lambda v: fmt_float(v, 4)),
    ("sd",       "Standard deviation",                    "qty",      lambda v: fmt_float(v, 4)),
]
values = dict(zip([m[0] for m in metric_meta], raw))

con.execute(f"""
CREATE OR REPLACE TEMP TABLE daily_unit_groups AS
SELECT
    ARTIKEL_ID,
    MARKT_ID,
    DATE,
    {DEMAND_COL},
    CASE
        WHEN COALESCE(GEWICHT_FLAG, 0) = 1 THEN 'gewicht_flag_1'
        ELSE 'gewicht_flag_0'
    END AS unit_group
FROM read_parquet('{PARQUET_GLOB}')
""")

summary_by_unit = con.execute(f"""
SELECT
    unit_group,
    COUNT(*)                                              AS n_rows,
    COUNT(DISTINCT ARTIKEL_ID)                            AS n_articles,
    COUNT(DISTINCT (ARTIKEL_ID, MARKT_ID))                AS n_series,
    MIN({DEMAND_COL})                                     AS min,
    quantile_cont({DEMAND_COL}, 0.50)                     AS p50,
    quantile_cont({DEMAND_COL}, 0.90)                     AS p90,
    quantile_cont({DEMAND_COL}, 0.99)                     AS p99,
    quantile_cont({DEMAND_COL}, 0.999)                    AS p999,
    MAX({DEMAND_COL})                                     AS max,
    AVG({DEMAND_COL})                                     AS mean,
    stddev_samp({DEMAND_COL})                             AS sd
FROM daily_unit_groups
GROUP BY unit_group
ORDER BY CASE unit_group
    WHEN 'gewicht_flag_1' THEN 1
    WHEN 'gewicht_flag_0' THEN 2
    ELSE 3 END
""").fetchdf()

summary_display = summary_by_unit.copy()
summary_display["unit_group"] = summary_display["unit_group"].map(UNIT_GROUP_LABELS)
for col in ["n_rows", "n_articles", "n_series"]:
    summary_display[col] = summary_display[col].map(fmt_int)
for col in ["min", "p50", "p90", "p99", "p999", "max", "mean", "sd"]:
    summary_display[col] = summary_display[col].map(lambda v: fmt_float(v, 4))

metric_rows = [
    {"metric": label, "unit": unit, "value": formatter(values[name])}
    for name, label, unit, formatter in metric_meta
]
print("Overall distribution summary")
display(pd.DataFrame(metric_rows))
print("Distribution summary by GEWICHT_FLAG")
summary_display


In [ ]:
def hist_via_duckdb_group(unit_group, expr, lo, hi, n_bins=80):
    """Aggregate histogram counts directly in duckdb to avoid huge in-memory arrays."""
    width = (hi - lo) / n_bins
    escaped_group = unit_group.replace("'", "''")
    df = con.execute(f"""
        SELECT
            FLOOR(({expr} - {lo}) / {width}) AS bin_idx,
            COUNT(*) AS n
        FROM daily_unit_groups
        WHERE unit_group = '{escaped_group}'
          AND {expr} BETWEEN {lo} AND {hi}
        GROUP BY bin_idx
        ORDER BY bin_idx
    """).fetchdf()
    edges = np.linspace(lo, hi, n_bins + 1)
    counts = np.zeros(n_bins, dtype=np.int64)
    if not df.empty:
        idx = df["bin_idx"].clip(0, n_bins - 1).astype(int).to_numpy()
        counts[idx] = df["n"].to_numpy()
    return edges, counts

p999_by_group = dict(con.execute(f"""
    SELECT unit_group, quantile_cont({DEMAND_COL}, 0.999) AS p999
    FROM daily_unit_groups
    GROUP BY unit_group
""").fetchall())

available_unit_groups = [
    unit_group
    for unit_group in UNIT_GROUP_ORDER
    if unit_group in p999_by_group and pd.notna(p999_by_group[unit_group])
]

if not available_unit_groups:
    raise ValueError("No unit groups available for distribution plots.")

fig, axes = plt.subplots(
    len(available_unit_groups),
    2,
    figsize=(12, 3.2 * len(available_unit_groups)),
    sharey=False,
    squeeze=False,
)
for row_idx, unit_group in enumerate(available_unit_groups):
    label = UNIT_GROUP_LABELS[unit_group]
    p999_group = float(p999_by_group[unit_group])

    edges_lin, counts_lin = hist_via_duckdb_group(unit_group, DEMAND_COL, 0.0, p999_group)
    edges_log, counts_log = hist_via_duckdb_group(
        unit_group, f"log10({DEMAND_COL} + 1e-6)", -6.0, 6.0
    )

    ax_lin, ax_log = axes[row_idx]
    ax_lin.bar(edges_lin[:-1], counts_lin, width=np.diff(edges_lin), align="edge")
    ax_lin.set_title(f"{label}: Tagesabsatz bis zum 99,9-%-Quantil")
    ax_lin.set_xlabel("Absatzmenge")
    ax_lin.set_ylabel("Anzahl Beobachtungen")

    ax_log.bar(edges_log[:-1], counts_log, width=np.diff(edges_log), align="edge", color="C1")
    ax_log.set_title(f"{label}: log10(Tagesabsatz)")
    ax_log.set_xlabel("log10(Absatzmenge)")
    ax_log.set_ylabel("Anzahl Beobachtungen")

plt.tight_layout()
plt.show()

### Top 20 largest values

Inspect the highest `ABVERKAUFTE_MENGE_KG` values to see whether the extreme tail is concentrated in a particular kind of article (e.g. weight-sold goods where a single mis-priced scale entry can blow up the recorded quantity).

In [ ]:
top_rows = con.execute(f"""
SELECT
    ARTIKEL_ID, MARKT_ID, DATE,
    {DEMAND_COL} AS qty,
    ARTIKEL_BEZ,
    VERKAUFSEINHEIT AS unit,
    GEWICHT_FLAG AS gewicht_flag
FROM read_parquet('{PARQUET_GLOB}')
ORDER BY {DEMAND_COL} DESC
LIMIT 20
""").fetchdf()
top_rows["qty"] = top_rows["qty"].map(lambda v: fmt_float(v, 4))
top_rows.index.name = "rank"
top_rows.index = top_rows.index + 1
print(f"Top 20 rows by {DEMAND_COL} (largest first)")
top_rows


## Per-series IQR bounds

For each `(ARTIKEL_ID, MARKT_ID)` series we compute:

- `Q1` - 25th percentile of `ABVERKAUFTE_MENGE_KG`,
- `Q3` - 75th percentile of `ABVERKAUFTE_MENGE_KG`,
- `IQR = Q3 - Q1`,
- `upper = Q3 + 3 * IQR` (Tukey "extreme outlier" cutoff, `k = 3`).

A row is flagged as outlier if its `ABVERKAUFTE_MENGE_KG > upper` *of its own series*. The split above is only for interpretability of descriptive plots. Bounds remain per series because products are sold in different units and series magnitudes differ by orders of magnitude.

In [ ]:
group_sql = ", ".join(GROUP_COLS)

con.execute(f"""
CREATE OR REPLACE TEMP TABLE bounds AS
WITH q AS (
    SELECT
        {group_sql},
        quantile_cont({DEMAND_COL}, 0.25) AS q1,
        quantile_cont({DEMAND_COL}, 0.75) AS q3
    FROM read_parquet('{PARQUET_GLOB}')
    GROUP BY {group_sql}
)
SELECT {group_sql}, q1, q3, q3 + {IQR_K} * (q3 - q1) AS upper
FROM q
""")

n_bounds = con.execute("SELECT COUNT(*) FROM bounds").fetchone()[0]
print(f"Bounds computed for {fmt_int(n_bounds)} series  (k={IQR_K}, upper = Q3 + k * IQR)")

In [ ]:
imp = con.execute(f"""
WITH joined AS (
    SELECT t.{DEMAND_COL}, b.upper
    FROM read_parquet('{PARQUET_GLOB}') t
    JOIN bounds b USING ({group_sql})
)
SELECT
    COUNT(*)                                                AS n_rows,
    COUNT(*) FILTER (WHERE {DEMAND_COL} > upper)            AS n_outliers,
    100.0 * COUNT(*) FILTER (WHERE {DEMAND_COL} > upper)
          / COUNT(*)                                        AS pct_outliers
FROM joined
""").fetchone()

impact = pd.DataFrame(
    [
        ("n_rows",       "Total rows in dataset",              fmt_int(imp[0])),
        ("n_outliers",   "Rows above Q3 + 3*IQR (per series)", fmt_int(imp[1])),
        ("n_kept",       "Rows kept after filtering",          fmt_int(imp[0] - imp[1])),
        ("pct_outliers", "Share of rows flagged (%)",          fmt_float(imp[2], 4)),
    ],
    columns=["metric", "description", "value"],
).set_index("metric")
print(f"Filtering impact at k = {IQR_K} (extreme-outlier Tukey cutoff)")
impact

### Per-series outlier counts

How is the outlier mass distributed across series? `series_with_outliers_pct` shows how widespread the issue is, the percentile rows show how many outliers a typical affected series has.

In [ ]:
per_series = con.execute(f"""
WITH joined AS (
    SELECT t.ARTIKEL_ID, t.MARKT_ID, t.{DEMAND_COL}, b.upper
    FROM read_parquet('{PARQUET_GLOB}') t
    JOIN bounds b USING ({group_sql})
)
SELECT
    ARTIKEL_ID, MARKT_ID,
    COUNT(*)                                        AS n_rows,
    COUNT(*) FILTER (WHERE {DEMAND_COL} > upper)    AS n_outliers
FROM joined
GROUP BY ARTIKEL_ID, MARKT_ID
""").fetchdf()

affected = per_series.loc[per_series.n_outliers > 0, "n_outliers"]
desc = affected.describe()

per_series_summary = pd.DataFrame(
    [
        ("series_total",            "All series",                              fmt_int(len(per_series))),
        ("series_with_outliers",    "Series with >= 1 outlier",                fmt_int(len(affected))),
        ("series_with_outliers_pct","Share of series with >= 1 outlier (%)",   fmt_float(100 * len(affected) / len(per_series), 2)),
        ("outliers_per_series_min", "Min outliers (within affected series)",   fmt_int(desc["min"])),
        ("outliers_per_series_p25", "25th percentile outliers per series",     fmt_int(desc["25%"])),
        ("outliers_per_series_p50", "Median outliers per series",              fmt_int(desc["50%"])),
        ("outliers_per_series_p75", "75th percentile outliers per series",     fmt_int(desc["75%"])),
        ("outliers_per_series_max", "Max outliers in a single series",         fmt_int(desc["max"])),
        ("outliers_per_series_mean","Mean outliers per affected series",       fmt_float(desc["mean"], 2)),
        ("outliers_per_series_std", "Std-dev outliers per affected series",    fmt_float(desc["std"], 2)),
    ],
    columns=["metric", "description", "value"],
).set_index("metric")
print("Per-series outlier counts (only series with at least one outlier are included in the percentile rows)")
per_series_summary

## Example series before vs after

Pick the series with the largest absolute outlier and the series with the most outliers, and show their daily quantity over time with the IQR upper bound.

In [ ]:
con.execute(f"""
CREATE OR REPLACE TEMP TABLE per_series_outliers AS
WITH joined AS (
    SELECT t.ARTIKEL_ID, t.MARKT_ID, t.{DEMAND_COL} AS qty, b.upper
    FROM read_parquet('{PARQUET_GLOB}') t
    JOIN bounds b USING ({group_sql})
)
SELECT
    ARTIKEL_ID, MARKT_ID,
    MAX(qty)                              AS max_qty,
    ANY_VALUE(upper)                      AS upper,
    MAX(qty) - ANY_VALUE(upper)           AS exceedance,
    COUNT(*) FILTER (WHERE qty > upper)   AS n_outliers
FROM joined
GROUP BY ARTIKEL_ID, MARKT_ID
HAVING COUNT(*) FILTER (WHERE qty > upper) > 0
""")

largest = con.execute("""
SELECT 'largest exceedance' AS reason, *
FROM per_series_outliers
ORDER BY exceedance DESC
LIMIT 1
""").fetchdf()

most = con.execute("""
SELECT 'most outliers' AS reason, *
FROM per_series_outliers
WHERE NOT (ARTIKEL_ID = ? AND MARKT_ID = ?)
ORDER BY n_outliers DESC
LIMIT 1
""", [int(largest.iloc[0]["ARTIKEL_ID"]), int(largest.iloc[0]["MARKT_ID"])]).fetchdf()

examples = pd.concat([largest, most], ignore_index=True)

examples_disp = examples.copy()
for col in ("max_qty", "upper", "exceedance"):
    examples_disp[col] = examples_disp[col].map(lambda v: fmt_float(v, 4))
examples_disp["n_outliers"] = examples_disp["n_outliers"].map(fmt_int)
print("Two series chosen for visualization (distinct series, one per criterion):")
examples_disp

In [ ]:
import re


def _clean_bez(name: str) -> str:
    """Strip digits from ARTIKEL_BEZ and collapse whitespace."""
    s = re.sub(r"\d+", "", str(name))
    return re.sub(r"\s+", " ", s).strip()


def plot_series(artikel_id: int, markt_id: int, ax,
                reason: str = "", markt_label: str = ""):
    df = con.execute(f"""
        SELECT CAST(t.DATE AS DATE) AS date,
               t.{DEMAND_COL}      AS qty,
               b.upper,
               t.ARTIKEL_BEZ       AS bez
        FROM read_parquet('{PARQUET_GLOB}') t
        JOIN bounds b USING ({group_sql})
        WHERE t.ARTIKEL_ID = ? AND t.MARKT_ID = ?
        ORDER BY date
    """, [artikel_id, markt_id]).fetchdf()
    upper = df["upper"].iloc[0]
    bez = _clean_bez(df["bez"].iloc[0])
    is_out = df["qty"] > upper
    ax.plot(df["date"], df["qty"], lw=1.0, color="C0", label="Tagesabsatz")
    ax.scatter(df.loc[is_out, "date"], df.loc[is_out, "qty"],
               color="C3", s=18, zorder=3, label="markierter Ausreißer")
    ax.axhline(upper, color="C1", ls="--", lw=1, label="serienspezifische Grenze")
    prefix = f"{reason}: " if reason else ""
    market = markt_label or f"Markt-ID={markt_id}"
    ax.set_title(f"{prefix}{bez}, {market}")
    ax.set_ylabel("Absatzmenge")
    ax.legend(loc="upper right", fontsize=8)

fig, axes = plt.subplots(len(examples), 1, figsize=(11, 3.2 * len(examples)),
                         sharex=False)
if len(examples) == 1:
    axes = [axes]
markt_labels = [f"Markt {chr(ord('A') + i)}" for i in range(len(examples))]
reason_labels = {
    "largest exceedance": "Größte Abweichung",
    "most outliers": "Viele markierte Tage",
}
for ax, (_, row), markt_label in zip(axes, examples.iterrows(), markt_labels):
    plot_series(int(row.ARTIKEL_ID), int(row.MARKT_ID), ax,
                reason=reason_labels.get(row.reason, row.reason),
                markt_label=markt_label)
plt.tight_layout()
plt.show()

## Cleaning

`src/data/cleaning/remove_outliers.py` applies the same per-series rule and writes the filtered yearly parquet files to the matching no-outlier directory: `data/interim/transactions_daily_agg_no_outliers/` for base data or `data/interim/transactions_daily_agg_fcm_no_outliers/` for FCM data.

### In-notebook check: before vs after

Apply the same per-series rule (`ABVERKAUFTE_MENGE_KG > Q3 + 3·IQR`) in memory and recompute the overall summary, then show the original ("before") and filtered ("after") numbers side-by-side. The "after" column is what the downstream cleaned parquet in the matching no-outlier directory will look like.

In [ ]:
cleaned_raw = con.execute(f"""
WITH filtered AS (
    SELECT t.{DEMAND_COL} AS qty, t.ARTIKEL_ID, t.MARKT_ID
    FROM read_parquet('{PARQUET_GLOB}') t
    JOIN bounds b USING ({group_sql})
    WHERE t.{DEMAND_COL} <= b.upper
)
SELECT
    COUNT(*)                                AS n_rows,
    COUNT(DISTINCT (ARTIKEL_ID, MARKT_ID))  AS n_series,
    MIN(qty)                                AS min,
    quantile_cont(qty, 0.50)                AS p50,
    quantile_cont(qty, 0.90)                AS p90,
    quantile_cont(qty, 0.99)                AS p99,
    quantile_cont(qty, 0.999)               AS p999,
    quantile_cont(qty, 0.9999)              AS p9999,
    MAX(qty)                                AS max,
    AVG(qty)                                AS mean,
    stddev_samp(qty)                        AS sd
FROM filtered
""").fetchone()

cleaned_values = dict(zip(
    ["n_rows","n_series","min","p50","p90","p99","p999","p9999","max","mean","sd"],
    cleaned_raw,
))

# `values` and `metric_meta` come from the overall-distribution cell above.
n_removed = values["n_rows"] - cleaned_values["n_rows"]
pct_removed = 100.0 * n_removed / values["n_rows"]

comparison = pd.DataFrame(
    [
        (name, desc, fmt(values[name]), fmt(cleaned_values[name]))
        for name, desc, _, fmt in metric_meta
    ],
    columns=["metric", "description", "before", "after"],
).set_index("metric")
print(f"Summary before vs after outlier removal (k = {IQR_K})")
print(f"Rows removed: {fmt_int(n_removed)}  ({fmt_float(pct_removed, 4)} %)")
print(f"Series lost (had only outlier rows): "
      f"{fmt_int(values['n_series'] - cleaned_values['n_series'])}")
comparison